# Collection Write Policy

**`CollectionWritePolicy`** (plugin_id `write_policy`) controls what happens when an
ingested feature collides with an already-stored entity that shares the same identity.

It is driver-agnostic: PG, Elasticsearch, Iceberg, and DuckDB all read the same config
from the waterfall (`collection → catalog → platform → code default`).

## `WriteConflictPolicy` values

| Value | Behaviour |
|---|---|
| `update` | Overwrite the existing entity in place (**default**) |
| `new_version` | Archive the old version; insert the new one as current |
| `refuse` | Skip this entity; continue processing the rest of the batch |
| `refuse_ingestion` | Hard-stop — reject the entire ingestion batch |

## Optional time-based versioning fields

| Field | Default | Purpose |
|---|---|---|
| `enable_validity` | `False` | Track `valid_from` / `valid_to` per entity |
| `validity_field` | `valid_from` | Source field to extract validity start from |
| `track_asset_id` | `True` | Stamp each entity with the ingestion `asset_id` |
| `external_id_field` | `id` | Dot-path used to extract the external identity key |
| `require_external_id` | `False` | Reject entities that lack an external_id |

Drivers that declare `Capability.TEMPORAL_VALIDITY` honour `enable_validity`.

In [ ]:
import httpx

BASE = "http://localhost:8000"
CATALOG_ID = "demo-write-policy"

client = httpx.AsyncClient(base_url=BASE, timeout=30)

# Create catalog once
r = await client.post("/stac/catalogs", json={"id": CATALOG_ID, "title": "Write Policy Demo"})
print("Catalog:", r.status_code)

---
## 1. `update` — overwrite in place (default)

Re-ingesting the same feature replaces the stored record.  Use when the source
of truth is always the latest snapshot (e.g. sensor readings, daily reports).

In [ ]:
COLL_UPDATE = "sensors-update"

r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLL_UPDATE,
    "title": "Sensors — UPDATE policy",
    "license": "proprietary",
})
print("Collection:", r.status_code)

# Set write policy
policy = {"on_conflict": "update", "track_asset_id": True, "external_id_field": "id"}
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLL_UPDATE}/configs/write_policy",
    json=policy,
)
print("Policy:", r.status_code)

In [ ]:
feature = {
    "type": "Feature",
    "id": "sensor-001",
    "geometry": {"type": "Point", "coordinates": [12.49, 41.89]},
    "properties": {"temperature": 22.3, "status": "nominal"},
}

# First ingest
r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_UPDATE}/items",
    json=feature,
)
print("First ingest:", r.status_code)

# Re-ingest with updated temperature — overwrites
feature["properties"]["temperature"] = 25.1
r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_UPDATE}/items",
    json=feature,
)
print("Re-ingest (update):", r.status_code)

r = await client.get(f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_UPDATE}/items/sensor-001")
item = r.json()
print("Stored temperature:", item["properties"]["temperature"])  # 25.1

---
## 2. `new_version` — temporal versioning

Each re-ingest creates a new temporal row; the previous version is archived.
Use for datasets where history matters (regulations, auditing, change detection).

Enable `enable_validity` to track the `valid_from` / `valid_to` window
extracted from the feature payload.

In [ ]:
COLL_VERSION = "land-use-versioned"

r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLL_VERSION,
    "title": "Land Use — NEW_VERSION policy",
    "license": "CC-BY-4.0",
})
print("Collection:", r.status_code)

policy = {
    "on_conflict": "new_version",
    "enable_validity": True,
    "validity_field": "reference_year",
    "external_id_field": "properties.parcel_id",
}
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLL_VERSION}/configs/write_policy",
    json=policy,
)
print("Policy:", r.status_code)

In [ ]:
parcel = {
    "type": "Feature",
    "geometry": {"type": "Polygon", "coordinates": [[[12.0, 41.0], [12.1, 41.0], [12.1, 41.1], [12.0, 41.1], [12.0, 41.0]]]},
    "properties": {"parcel_id": "P-00123", "land_class": "arable", "reference_year": "2022"},
}

# 2022 version
r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_VERSION}/items", json=parcel,
)
print("2022 ingest:", r.status_code)

# 2024 reclassification — creates new version, archives 2022
parcel["properties"]["land_class"] = "forest"
parcel["properties"]["reference_year"] = "2024"
r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_VERSION}/items", json=parcel,
)
print("2024 ingest (new version):", r.status_code)

# Browse all versions
r = await client.get(f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_VERSION}/items?limit=10")
items = r.json().get("features", [])
print(f"Stored versions: {len(items)}")
for it in items:
    p = it["properties"]
    print(f"  parcel={p.get('parcel_id')} class={p.get('land_class')} year={p.get('reference_year')}")

---
## 3. `refuse` — skip duplicate, continue batch

If the identity already exists, the incoming entity is silently discarded.
The rest of the batch continues normally.  Use for idempotent pipelines where
duplicate delivery must not overwrite authoritative records.

In [ ]:
COLL_REFUSE = "boundaries-refuse"

r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLL_REFUSE, "title": "Boundaries — REFUSE policy", "license": "ODbL",
})
print("Collection:", r.status_code)

policy = {"on_conflict": "refuse", "external_id_field": "properties.iso3"}
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLL_REFUSE}/configs/write_policy",
    json=policy,
)
print("Policy:", r.status_code)

batch = {"type": "FeatureCollection", "features": [
    {"type": "Feature", "geometry": {"type": "Point", "coordinates": [12.5, 41.9]},
     "properties": {"iso3": "ITA", "name": "Italy"}},
    {"type": "Feature", "geometry": {"type": "Point", "coordinates": [2.3, 48.9]},
     "properties": {"iso3": "FRA", "name": "France"}},
]}

r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_REFUSE}/items", json=batch,
)
print("First batch:", r.status_code)

# Attempt to re-ingest Italy + add Germany — Italy is refused, Germany written
batch2 = {"type": "FeatureCollection", "features": [
    {"type": "Feature", "geometry": {"type": "Point", "coordinates": [12.5, 41.9]},
     "properties": {"iso3": "ITA", "name": "Italy MODIFIED"}},  # duplicate → refused
    {"type": "Feature", "geometry": {"type": "Point", "coordinates": [10.4, 51.2]},
     "properties": {"iso3": "DEU", "name": "Germany"}},          # new → written
]}
r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_REFUSE}/items", json=batch2,
)
print("Second batch (Italy refused, Germany written):", r.status_code)

r = await client.get(f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_REFUSE}/items")
items = r.json().get("features", [])
print(f"Total stored: {len(items)} (ITA unchanged, DEU added)")
for it in items:
    print("  ", it["properties"]["iso3"], "-", it["properties"]["name"])

---
## 4. `refuse_ingestion` — hard-stop on duplicate

The first collision aborts the entire batch with an error.  Use for high-integrity
pipelines (financial records, legal boundaries) where accidental duplicates must
never go unnoticed.

In [ ]:
COLL_HARD = "parcels-strict"

r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLL_HARD, "title": "Parcels — REFUSE_INGESTION policy", "license": "proprietary",
})
print("Collection:", r.status_code)

policy = {
    "on_conflict": "refuse_ingestion",
    "require_external_id": True,
    "external_id_field": "properties.cadastral_id",
}
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLL_HARD}/configs/write_policy",
    json=policy,
)
print("Policy:", r.status_code)

parcel = {"type": "Feature",
          "geometry": {"type": "Point", "coordinates": [11.2, 43.8]},
          "properties": {"cadastral_id": "CAD-9912", "area_m2": 5200}}

r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_HARD}/items", json=parcel,
)
print("First ingest:", r.status_code)

# Duplicate → entire batch rejected with 409
r = await client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{COLL_HARD}/items", json=parcel,
)
print("Duplicate ingest — expected 409:", r.status_code)
if r.status_code == 409:
    print("  ✓ Batch correctly rejected:", r.json().get("detail", r.text[:100]))

---
## Cascade: collection-level policy overrides catalog default

The config waterfall means you can set a platform/catalog default and override
at collection level for specific datasets.

In [ ]:
# Set catalog-level default: refuse (protect all collections by default)
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/configs/write_policy",
    json={"on_conflict": "refuse"},
)
print("Catalog default (refuse):", r.status_code)

# Override one collection to allow updates
COLL_OVERRIDE = "snapshots-update"
r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLL_OVERRIDE, "title": "Snapshots (update override)", "license": "CC-BY-4.0",
    "write_policy": {"on_conflict": "update"},  # embedded at creation
})
print("Collection with embedded write_policy:", r.status_code)

In [ ]:
# Cleanup
r = await client.delete(f"/stac/catalogs/{CATALOG_ID}?force=true")
print("Cleanup:", r.status_code)
await client.aclose()